# PARC2026 — D10 π0.5 Dataset Top-2 Tie-break (L4/A100)

`V1_MULTI` と `V2_SQRT` が初回固定Simulator評価で **17/32 = 0.53125** の同点だったため、
同じ2つの150-step LoRA adapterを **独立eval seed** で再評価します。

- 初回: seed `20260905`
- Tie-break: seed `20260906`
- Track1 / 4 tasks / 8 episodes per task / max 300 steps
- **学習lossでは決めない**
- 結果が再度同点なら自動promoteしない
- L4 24GB推奨（本番推論条件に近い）
- 実行中は親プロセスが30秒heartbeatを出します
- 実結果はDrive `pi05-top2-tiebreak-v1/` に保存します

既存 `60_pi05_fixed_sim_eval_l4.ipynb` を一時コピーして、
variant・seed・Drive出力先だけを機械的に差し替えて実行するため、
Simulator/merge/evalの実装を二重管理しません。


In [ ]:
# 0/4 Fresh-runtime preflight
import os, json, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

print("=== 0/4 D10 TIE-BREAK PREFLIGHT ===", flush=True)
gpu = subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader,nounits"], text=True).strip()
print("GPU:", gpu, flush=True)
if "T4" in gpu.upper(): raise RuntimeError("T4は対象外です。L4またはA100を使用してください。")
free = shutil.disk_usage("/content").free / 1024**3
print(f"local free: {free:.1f} GiB", flush=True)
if free < 45: raise RuntimeError("fresh runtime推奨: local free >=45 GiBで再実行してください。")
drive.mount("/content/drive")
try:
    tok = os.environ.get("HF_TOKEN") or userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Colab Secretsに HF_TOKEN を登録し、Notebook accessをONにしてください。") from e
if not tok: raise RuntimeError("HF_TOKEN is empty")
os.environ["HF_TOKEN"] = tok
DRIVE = Path("/content/drive/MyDrive/parc2026-cache")
BASE = DRIVE / "pi05-fixed-sim-eval-v2" / "screening_summary.json"
ABLAT = DRIVE / "pi05-ablation-group-aware-v2"
assert BASE.exists(), BASE
base = json.loads(BASE.read_text())
scores = {r["variant"]: r.get("overall_score") for r in base.get("results", [])}
assert scores.get("V1_MULTI") == 0.53125, scores
assert scores.get("V2_SQRT") == 0.53125, scores
for run in ["colab_data_v1_multi", "colab_data_v2_sqrt"]:
    assert (ABLAT/run/"pretrained_model").is_dir(), run
print("BASE tie:", {"V1_MULTI":scores["V1_MULTI"], "V2_SQRT":scores["V2_SQRT"]})
print("HF_TOKEN: FOUND (hidden)")
print("=== PREFLIGHT: PASS ===")


In [ ]:
# 1/4 Clone current repo and generate the execution notebook from notebook 60
import json, subprocess
from pathlib import Path
ROOT = Path("/content/parc2026"); REPO = ROOT / "py_AI"; ROOT.mkdir(parents=True, exist_ok=True)
if not (REPO/".git").exists(): subprocess.run(["git","clone","https://github.com/yu37330/py_AI.git",str(REPO)], check=True)
subprocess.run(["git","-C",str(REPO),"fetch","origin","main"], check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--force","origin/main"], check=True)
src = REPO / "colab/60_pi05_fixed_sim_eval_l4.ipynb"; assert src.exists(), src
raw = json.loads(src.read_text())
for cell in raw["cells"]:
    text = "".join(cell.get("source", []))
    text = text.replace("pi05-fixed-sim-eval-v2", "pi05-top2-tiebreak-v1").replace("20260905", "20260906")
    text = text.replace("tok = userdata.get('HF_TOKEN')", "tok = os.environ.get('HF_TOKEN') or userdata.get('HF_TOKEN')")
    lines = text.splitlines(True)
    lines = [ln for ln in lines if "('V0_RAW','colab_data_v0_raw')" not in ln and "('V1_ALL_REVIEW','colab_data_v1_all_review')" not in ln]
    cell["source"] = lines
target = Path("/content/pi05_top2_tiebreak_exec.ipynb")
target.write_text(json.dumps(raw, ensure_ascii=False, indent=1))
text_all = target.read_text()
assert "colab_data_v1_multi" in text_all and "colab_data_v2_sqrt" in text_all
assert "colab_data_v0_raw" not in text_all and "colab_data_v1_all_review" not in text_all
assert "20260906" in text_all and "pi05-top2-tiebreak-v1" in text_all
print("generated:", target)
print("=== EXECUTION NOTEBOOK CONTRACT: PASS ===")


In [ ]:
# 2/4 Execute unattended with parent heartbeat
import os, shutil, subprocess, threading, time
from collections import deque
from pathlib import Path
target = Path("/content/pi05_top2_tiebreak_exec.ipynb"); executed = Path("/content/pi05_top2_tiebreak_executed.ipynb")
def gpu_status():
    try:
        raw = subprocess.check_output(["nvidia-smi","--query-gpu=utilization.gpu,memory.used,memory.total","--format=csv,noheader,nounits"], text=True).strip()
        return [x.strip() for x in raw.split(",")]
    except Exception: return ["?","?","?"]
cmd = ["jupyter","nbconvert","--to","notebook","--execute",str(target),"--output",str(executed),"--ExecutePreprocessor.timeout=-1"]
env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"; env["MPLBACKEND"] = "Agg"
print("=== 2/4 RUN TOP-2 TIE-BREAK ===", flush=True)
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
tail = deque(maxlen=150)
def reader():
    for line in iter(p.stdout.readline, ""):
        line=line.rstrip(); tail.append(line); print("[child]", line, flush=True)
t=threading.Thread(target=reader,daemon=True); t.start(); start=time.time()
while p.poll() is None:
    util,used,total=gpu_status(); elapsed=int(time.time()-start); free=shutil.disk_usage("/content").free/1024**3
    print(f"[heartbeat] tie-break | elapsed={elapsed//3600:02d}:{(elapsed%3600)//60:02d}:{elapsed%60:02d} | GPU={util}% | VRAM={used}/{total} MiB | free={free:.1f} GiB | running", flush=True)
    time.sleep(30)
t.join(timeout=10); print("returncode:", p.returncode, flush=True)
if p.returncode != 0:
    print("=== LAST 150 LINES ==="); print("\n".join(tail)); raise RuntimeError("top-2 tie-break notebook failed")
print("=== TOP-2 TIE-BREAK EXECUTION: PASS ===")


In [ ]:
# 3/4 Readback, rank and freeze provisional dataset decision only if decisive
import json, subprocess
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/parc2026-cache"); OUT = DRIVE / "pi05-top2-tiebreak-v1"
summary_path = OUT / "screening_summary.json"; assert summary_path.exists(), summary_path
data = json.loads(summary_path.read_text())
scores = {r["variant"]: r.get("overall_score") for r in data.get("results", []) if r.get("variant") in {"V1_MULTI","V2_SQRT"}}
assert set(scores) == {"V1_MULTI","V2_SQRT"}, scores; assert all(v is not None for v in scores.values()), scores
ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
print("=== TIE-BREAK RESULT ===")
for i,(name,score) in enumerate(ranked,1): print(f"{i}. {name:12s} score={score}")
canonical = {"V1_MULTI": "V1_MULTI_FLAG_PRUNED_EXPERIMENTAL", "V2_SQRT": "V2_SQRT_BALANCED_RAW"}
if ranked[0][1] == ranked[1][1]:
    decision = {"status":"STILL_TIED","selected_variant":None,"scores":scores,"eval_seed":20260906,"note":"Do not auto-promote. Review task-level stability and pre-registered secondary evidence."}
else:
    decision = {"status":"DECIDED","selected_variant":canonical[ranked[0][0]],"selected_short_name":ranked[0][0],"scores":scores,"eval_seed":20260906,"decision_rule":"simulator_success_rate_primary"}
decision["repo_sha"] = subprocess.check_output(["git","-C","/content/parc2026/py_AI","rev-parse","HEAD"], text=True).strip()
decision_path = OUT / "provisional_best_dataset_recipe.json"; decision_path.write_text(json.dumps(decision, indent=2)+"\n")
print(json.dumps(decision, indent=2)); print("saved:",decision_path); print("=== D10 COMPLETE ===")
